# Non-attack undersampling by cosine similarity

Applies the rule-based filters, then keeps the 2,379 non-attack rows with the highest mean cosine similarity plus all attack rows, and saves `train_sampled_data.csv`.

In [ ]:
DATA_DIR = "../data"

In [ ]:
import time
import umap
import umap.plot
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
import keras.backend as K

from transformers import BertTokenizer, TFBertModel

from catboost import CatBoostClassifier

from nlp_id.lemmatizer import Lemmatizer
from nlp_id.tokenizer import Tokenizer, PhraseTokenizer
from nlp_id.postag import PosTag
from nlp_id.stopword import StopWord

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold
from sklearn import feature_selection as fs
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn import preprocessing
from sklearn.metrics.pairwise import cosine_similarity

from scipy import sparse

from sentence_transformers import SentenceTransformer

from IPython.display import IFrame
from IPython.core.display import display, HTML

/opt/conda/envs/tf-torch/lib/python3.10/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/opt/conda/envs/tf-torch/lib/python3.10/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/opt/conda/envs/tf-torch/lib/python3.10/site-packages/umap/distances.py:1086: NumbaDepreca

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
train

,Unnamed: 0,id,timestamp,remote_addr,remote_user,hostname,time_local,request,status,body_bytes_sent,...,upstream_response_time,upstream_addr,upstream_status,connection,connection_requests,data.srcip,agent.ip,rule.mitre.tactic,hour,pairwise
0,0,38,2023-08-29 17:05:43.445,66.96.233.60,-,vm-director-nginx,30/Aug/2023:00:05:42 +0700,POST /lib/ajax/service.php?sesskey=FyapD11LyJ&...,200,98,...,0.03,10.45.184.157:32401,200,688312591,181,66.96.233.60,192.168.1.99,"[""Impact""]",17,0.126820
1,1,54,2023-08-29 17:08:23.939,118.99.125.4,-,vm-director-nginx,30/Aug/2023:00:08:23 +0700,POST /lib/ajax/service.php?sesskey=QY0HDC3szQ&...,200,1169,...,0.068,10.45.184.143:32401,200,688315946,158,118.99.125.4,192.168.1.99,"[""Impact""]",17,0.091207
2,2,98,2023-08-29 17:16:56.322,125.166.1.20,-,vm-director-nginx,30/Aug/2023:00:16:54 +0700,POST /lib/ajax/service.php?sesskey=9dbAEW9y6f&...,200,27,...,0.088,10.45.184.157:32401,200,688328194,209,125.166.1.20,192.168.1.99,"[""Impact""]",17,0.290300
3,3,99,2023-08-29 17:17:14.250,103.171.147.123,-,vm-director-nginx,30/Aug/2023:00:17:13 +0700,POST /lib/ajax/service.php?sesskey=nscrep2nd3&...,200,27,...,0.098,10.45.184.157:32401,200,688330190,90,103.171.147.123,192.168.1.99,"[""Impact""]",17,0.328679
4,4,187,2023-08-29 17:22:58.771,180.248.17.197,-,vm-director-nginx,30/Aug/2023:00:22:56 +0700,POST /lib/ajax/service.php?sesskey=ezddPBRNs5&...,200,6712,...,0.104,10.45.184.142:32401,200,688336653,219,180.248.17.197,192.168.1.99,"[""Impact""]",17,0.077081
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4753,4753,10060,2023-08-30 23:07:44.775,103.171.147.86,-,vm-director-nginx,31/Aug/2023:06:07:42 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,...,0.002,10.45.184.154:32442,404,694907441,12,103.171.147.86,192.168.1.99,"[""non-attack""]",23,0.983679
4754,4754,10187,2023-08-30 23:26:20.011,202.80.217.85,-,vm-director-nginx,31/Aug/2023:06:26:19 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,...,0.002,10.45.184.154:32442,404,694944826,2,202.80.217.85,192.168.1.99,"[""non-attack""]",23,0.983679
4755,4755,6735,2023-08-30 00:45:22.206,149.113.53.254,-,vm-director-nginx,30/Aug/2023:07:45:21 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,...,0.002,10.45.184.154:32442,404,689249474,4,149.113.53.254,192.168.1.99,"[""non-attack""]",0,0.983679
4756,4756,10177,2023-08-30 23:26:22.020,182.4.134.119,-,vm-director-nginx,31/Aug/2023:06:26:21 +0700,GET /auth/resources/ofl9o/login/ub/ HTTP/2.0,404,118,...,0.002,10.45.184.148:32442,404,694944919,2,182.4.134.119,192.168.1.99,"[""non-attack""]",23,0.983679


In [ ]:
train.columns

Index(['Unnamed: 0', 'id', 'timestamp', 'remote_addr', 'remote_user',
       'hostname', 'time_local', 'request', 'status', 'body_bytes_sent',
       'http_referer', 'http_user_agent', 'http_x_forwarded_for',
       'time_iso8601', 'request_method', 'request_length', 'bytes_sent',
       'http_referer2', 'request_time', 'upstream_connect_time',
       'upstream_header_time', 'upstream_response_time', 'upstream_addr',
       'upstream_status', 'connection', 'connection_requests', 'data.srcip',
       'agent.ip', 'rule.mitre.tactic', 'hour', 'pairwise'],
      dtype='object')

In [ ]:
#Time local
train['timestamp'] = pd.to_datetime(train['timestamp'], format='%b %d, %Y @ %H:%M:%S.%f')

# Create a new column 'hour' containing the hour component
train['hour'] = train['timestamp'].dt.hour

ValueError: time data '2023-08-29 17:05:43.445' does not match format '%b %d, %Y @ %H:%M:%S.%f' (match)

In [ ]:
train = train[~train['status'].isin([405, 407, 413, 422, 500, 502, 504])]

In [ ]:
cross_tab = pd.crosstab(train['http_referer'], train['rule.mitre.tactic'])
cross_tab
# Filter rows where only the 'non attack' column has values greater than 0
cross_tab[(cross_tab['["non-attack"]'] > 0) & (cross_tab.drop(columns='["non-attack"]').eq(0).all(axis=1))]
# Filter rows where only the 'non attack' column has values greater than 0
non_attack_http = cross_tab[(cross_tab['["non-attack"]'] > 0) & (cross_tab.drop(columns='["non-attack"]').eq(0).all(axis=1))]
non_attack_http_list = non_attack_http.index.tolist()

for http_referer in non_attack_http_list:
    print(http_referer)


binance.com
http://175.45.184.147:80/actuator/gateway/routes
http://175.45.184.193:80/
http://175.45.184.30/b1a3k.php
http://175.45.184.51/_phpMyAdmin/index.php?lang=en
http://175.45.184.51/_phpmyadmin_/index.php?lang=en
http://175.45.184.51/administrator/admin/index.php?lang=en
http://175.45.184.51/administrator/phpmyadmin/index.php?lang=en
http://175.45.184.51/database/index.php?lang=en
http://175.45.184.51/db/db-admin/index.php?lang=en
http://175.45.184.51/db/dbadmin/index.php?lang=en
http://175.45.184.51/db/phpMyAdmin-4/index.php?lang=en
http://175.45.184.51/dbadmin/index.php?lang=en
http://175.45.184.51/myadmin/index.php?lang=en
http://175.45.184.51/mysql-admin/index.php?lang=en
http://175.45.184.51/mysql/admin/index.php?lang=en
http://175.45.184.51/mysql/db/index.php?lang=en
http://175.45.184.51/mysql/dbadmin/index.php?lang=en
http://175.45.184.51/mysqladmin/index.php?lang=en
http://175.45.184.51/phpMyAdmin-3/index.php?lang=en
http://175.45.184.51/phpMyAdmin-5.1.1/index.php?lang=

In [ ]:
train = train[~train['http_referer'].isin(non_attack_http_list)]

In [ ]:
train = train[~train['request_method'].isin(["HEAD"])]

In [ ]:
train = train[~train['remote_user'].isin(["GapuraUb2020-Mobile", "google-oauth-client"])]

In [ ]:
cross_tab = pd.crosstab(train['upstream_addr'], train['rule.mitre.tactic'])

# Filter rows where only the 'non attack' column has values greater than 0
cross_tab[(cross_tab['["non-attack"]'] > 0) & (cross_tab.drop(columns='["non-attack"]').eq(0).all(axis=1))]
# Filter rows where only the 'non attack' column has values greater than 0
upstream_addr_nonattack = cross_tab[(cross_tab['["non-attack"]'] > 0) & (cross_tab.drop(columns='["non-attack"]').eq(0).all(axis=1))]
upstream_addr_list = upstream_addr_nonattack.index.tolist()
upstream_addr_list

['175.45.184.137:80',
 '175.45.184.237:80',
 '175.45.184.241:80',
 '175.45.184.242:80',
 '175.45.184.243:80']

In [ ]:
train = train[~train['upstream_addr'].isin(upstream_addr_list)]

In [ ]:
# Drop useless column
train_drop = train.drop(['timestamp', 'remote_addr', 'remote_user', 'hostname', 'time_local', 'request', 'http_referer', 'http_user_agent', 'http_x_forwarded_for', 'time_iso8601', 'http_referer2', 'upstream_addr', 'upstream_status', 'connection', 'data.srcip', 'agent.ip','rule.mitre.tactic'], axis=1)

In [ ]:
train_drop.columns

Index(['id', 'status', 'body_bytes_sent', 'request_method', 'request_length',
       'bytes_sent', 'request_time', 'upstream_connect_time',
       'upstream_header_time', 'upstream_response_time', 'connection_requests',
       'hour'],
      dtype='object')

In [ ]:
train_drop.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 52769 entries, 0 to 54479
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      52769 non-null  int64  
 1   status                  52769 non-null  int64  
 2   body_bytes_sent         52769 non-null  int64  
 3   request_method          0 non-null      float64
 4   request_length          52769 non-null  int64  
 5   bytes_sent              52769 non-null  int64  
 6   request_time            52769 non-null  float64
 7   upstream_connect_time   52676 non-null  float64
 8   upstream_header_time    52676 non-null  float64
 9   upstream_response_time  52676 non-null  float64
 10  connection_requests     52769 non-null  int64  
 11  hour                    52769 non-null  int64  
dtypes: float64(5), int64(7)
memory usage: 5.2 MB


In [ ]:
from sklearn.preprocessing import LabelEncoder
# Create a LabelEncoder instance
encoder = LabelEncoder()

# Fit and transform the 'request_method' column
train_drop['request_method'] = encoder.fit_transform(train_drop['request_method'])


In [ ]:
train_drop.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 52769 entries, 0 to 54479
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      52769 non-null  int64  
 1   status                  52769 non-null  int64  
 2   body_bytes_sent         52769 non-null  int64  
 3   request_method          52769 non-null  int64  
 4   request_length          52769 non-null  int64  
 5   bytes_sent              52769 non-null  int64  
 6   request_time            52769 non-null  float64
 7   upstream_connect_time   52676 non-null  float64
 8   upstream_header_time    52676 non-null  float64
 9   upstream_response_time  52676 non-null  float64
 10  connection_requests     52769 non-null  int64  
 11  hour                    52769 non-null  int64  
dtypes: float64(4), int64(8)
memory usage: 5.2 MB


In [ ]:
# Loop through each column
for column in train_drop.columns:
    train_drop[column] = pd.to_numeric(train_drop[column], errors='coerce')

# 'coerce' option will convert non-numeric values to NaN if they can't be converted
# If you want to drop rows with NaN values, you can use df.dropna()

print(train_drop.dtypes)

AttributeError: 'float' object has no attribute 'split'

In [ ]:
import pandas as pd

# Assuming you have your DataFrame already loaded, if not, load your data here

# Calculate the mean for each numeric column
mean_values = train_drop.mean()

# Fill missing values with the mean for each column
train_drop.fillna(mean_values, inplace=True)

# Display the DataFrame with missing values filled with means
train_drop.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 52769 entries, 0 to 54479
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      52769 non-null  int64  
 1   status                  52769 non-null  int64  
 2   body_bytes_sent         52769 non-null  int64  
 3   request_method          52769 non-null  int64  
 4   request_length          52769 non-null  int64  
 5   bytes_sent              52769 non-null  int64  
 6   request_time            52769 non-null  float64
 7   upstream_connect_time   52769 non-null  float64
 8   upstream_header_time    52769 non-null  float64
 9   upstream_response_time  52769 non-null  float64
 10  connection_requests     52769 non-null  int64  
 11  hour                    52769 non-null  int64  
dtypes: float64(4), int64(8)
memory usage: 5.2 MB


In [ ]:
#reduced_features_sparse = sparse.csr_matrix(all_features)
reduced_features_sparse = sparse.csr_matrix(train_drop_drop)
similarities = cosine_similarity(reduced_features_sparse)

#print(similarities.shape)
pairwise = np.mean(similarities, axis = 1)

In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 52769 entries, 0 to 54479
Data columns (total 29 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   id                      52769 non-null  int64         
 1   timestamp               52769 non-null  datetime64[ns]
 2   remote_addr             52769 non-null  object        
 3   remote_user             52769 non-null  object        
 4   hostname                52769 non-null  object        
 5   time_local              52769 non-null  object        
 6   request                 52769 non-null  object        
 7   status                  52769 non-null  int64         
 8   body_bytes_sent         52769 non-null  int64         
 9   http_referer            52769 non-null  object        
 10  http_user_agent         52769 non-null  object        
 11  http_x_forwarded_for    52769 non-null  object        
 12  time_iso8601            52769 non-null  object

In [ ]:
train['pairwise'] = pairwise

In [ ]:
non_attack = train[train['rule.mitre.tactic'] == '["non-attack"]']

In [ ]:
non_attack_sorted = non_attack.sort_values(['pairwise'], ascending=False)

In [ ]:
train['rule.mitre.tactic'].value_counts()

["non-attack"]           50390
["Impact"]                1784
["Reconnaissance"]         561
["Defense Evasion"]         29
["Initial Access"]           4
["Credential Access"]        1
Name: rule.mitre.tactic, dtype: int64

In [ ]:
non_attack_sample = non_attack_sorted[:2379]

In [ ]:
non_attack_sample.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2379 entries, 10907 to 10127
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   id                      2379 non-null   int64         
 1   timestamp               2379 non-null   datetime64[ns]
 2   remote_addr             2379 non-null   object        
 3   remote_user             2379 non-null   object        
 4   hostname                2379 non-null   object        
 5   time_local              2379 non-null   object        
 6   request                 2379 non-null   object        
 7   status                  2379 non-null   int64         
 8   body_bytes_sent         2379 non-null   int64         
 9   http_referer            2379 non-null   object        
 10  http_user_agent         2379 non-null   object        
 11  http_x_forwarded_for    2379 non-null   object        
 12  time_iso8601            2379 non-null   obj

In [ ]:
filtered_train = train[train['rule.mitre.tactic'] != '["non-attack"]']

In [ ]:
sampled = pd.concat([filtered_train, non_attack_sample], ignore_index=True)

In [ ]:
sampled.to_csv(f"{DATA_DIR}/train_sampled_data.csv")